# Diabetic Retinopathy Stage Detection with Transfer Learning

**Module:** Computer Vision, BSc (Hons) Computer Science, NIBM / Coventry University
**Coursework 1:** Diabetic Retinopathy Stage Detection

This Kaggle notebook runs the complete pipeline end to end and produces every figure and number used in the report.

| Step | Rubric criterion | Section |
|---|---|---|
| Dataset exploration, class balance, stratified split | 1. Problem understanding & dataset | 3 |
| Border crop, resize, CLAHE, Ben Graham, edge enhancement | 2. Preprocessing | 4 |
| On-the-fly augmentation and class weights | 3. Augmentation & balancing | 5 |
| EfficientNet transfer learning (plus 4 other backbones) | 4. CNN architecture | 6 |
| Two-phase training, early stopping, LR scheduling, checkpoints | 5. Training strategy | 6 |
| Curves, precision, recall, F1, QWK, confusion matrix, ROC, Grad-CAM | 6. Evaluation | 7 |
| Preprocessing and backbone comparison | 5 + 9. Experimental design, innovation | 8 |
| Results bundle for the report and the Streamlit demo | 7 + 8. Code, report | 9 |

### How to run on Kaggle
1. **Add Input:** search for `sovitrath/diabetic-retinopathy-224x224-2019-data` and add it.
2. **Settings:** Accelerator `GPU T4 x2`, and **Internet `On`**. Internet is required because the
   ImageNet weights of the pretrained CNNs are downloaded when each model is built.
   Both settings need a phone-verified Kaggle account.
3. Click **Save Version, Save & Run All (Commit)**. The run continues even if you close the browser.
   Main model: about 10 minutes. Full comparison grid (section 8): about 60 to 90 minutes more.
4. When the version finishes, open its **Output** tab and download `dr_results_bundle.zip`.

All logic lives in a small, commented Python package (`dr_detection`) that section 2 writes to disk.
The same package powers the command-line scripts and the Streamlit demo in the GitHub repository,
so the notebook only orchestrates it and displays the evidence.

## 1. Environment

In [ ]:
# ============================================================================
# 1.1  Detect the environment, choose working folders, check the GPU
# ============================================================================
import os, sys, json, time, shutil, pathlib, importlib, platform
from pathlib import Path

ON_KAGGLE = Path("/kaggle/input").exists()
# Local test hooks (ignored on Kaggle): DR_WORK_DIR, DR_INPUT_DIR, DR_QUICK_TEST
WORK_DIR  = Path("/kaggle/working") if ON_KAGGLE else Path(os.environ.get("DR_WORK_DIR", Path.cwd())).resolve()
INPUT_DIR = Path("/kaggle/input")   if ON_KAGGLE else Path(os.environ.get("DR_INPUT_DIR", WORK_DIR / "data")).resolve()
CACHE_DIR = Path("/tmp/dr_cache")   if ON_KAGGLE else WORK_DIR / "cache"   # /tmp is not saved as notebook output
QUICK_TEST = (not ON_KAGGLE) and os.environ.get("DR_QUICK_TEST", "0") == "1"

WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)
NOTEBOOK_START = time.time()

import tensorflow as tf, keras, cv2, sklearn, numpy as np, pandas as pd, matplotlib
print(f"Running on Kaggle: {ON_KAGGLE} | working dir: {WORK_DIR}")
print(f"Python {platform.python_version()} | TensorFlow {tf.__version__} | Keras {keras.__version__} | "
      f"OpenCV {cv2.__version__} | scikit-learn {sklearn.__version__} | NumPy {np.__version__}")
gpus = tf.config.list_physical_devices("GPU")
for g in gpus:
    print("GPU:", tf.config.experimental.get_device_details(g).get("device_name", g.name))
if ON_KAGGLE:
    assert gpus, ("No GPU detected. Open Settings -> Accelerator -> GPU T4 x2, then run again. "
                  "Training on the CPU would take many hours.")

# The pretrained ImageNet weights are downloaded from Google's storage bucket the first time
# each backbone is built, so the notebook needs internet access. Check it now, not after an hour.
import urllib.request
try:
    urllib.request.urlopen("https://storage.googleapis.com/tensorflow/keras-applications/", timeout=15)
    print("Internet: available (ImageNet weights can be downloaded)")
except urllib.error.HTTPError:
    print("Internet: available (ImageNet weights can be downloaded)")   # the bucket root answers 4xx, that is fine
except Exception as exc:
    raise RuntimeError("No internet access. Open Settings -> Internet -> On (requires a phone-verified "
                       "Kaggle account), then run again.") from exc

## 2. The `dr_detection` package

Each cell below writes one module of the package to `src/dr_detection/`. Keeping the code in modules
(rather than one long notebook) makes it testable, reusable by the Streamlit app, and easy to read
feature by feature. Every module starts with a docstring explaining its design decisions.

In [ ]:
PKG_DIR = WORK_DIR / "src" / "dr_detection"
PKG_DIR.mkdir(parents=True, exist_ok=True)
print("writing package to", PKG_DIR)

### `dr_detection/__init__.py`
Package marker and overview of the modules.

In [ ]:
%%writefile src/dr_detection/__init__.py
"""
dr_detection
============

A small, modular Python package for **Diabetic Retinopathy (DR) stage
detection** from colour fundus photographs using transfer learning.

The package is organised so that every step of the pipeline described in the
coursework brief lives in its own module:

* ``dr_detection.config``        - all tunable settings in one place
* ``dr_detection.preprocessing`` - border cropping, Ben Graham, CLAHE, edge enhancement
* ``dr_detection.data``          - dataset discovery, label mapping, stratified splits, tf.data
* ``dr_detection.augmentation``  - on-the-fly augmentation + class balancing helpers
* ``dr_detection.model``         - CNN backbones with transfer-learning heads
* ``dr_detection.train``         - two-phase training strategy with callbacks
* ``dr_detection.evaluate``      - metrics, confusion matrix, curves, error analysis
* ``dr_detection.gradcam``       - Grad-CAM explainability
* ``dr_detection.visualize``     - exploratory data analysis figures
* ``dr_detection.utils``         - seeding, I/O and timing helpers

The same code is used by the Colab notebook, the command-line scripts and the
Streamlit demo application, so results are reproducible everywhere.
"""

__version__ = "1.0.0"
__author__ = "BSCCOMP24.2P - Computer Vision Coursework"

### `dr_detection/config.py`
All hyper-parameters, class names, label spellings and dataset constants in one place.

In [ ]:
%%writefile src/dr_detection/config.py
"""
Central configuration for the DR stage detection project.

Everything that a user might want to tune (image size, epochs, learning
rates, backbone, preprocessing method, paths) is defined here so that the
notebook, the CLI scripts and the Streamlit app all read the same values.
"""

from __future__ import annotations

from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List

# --------------------------------------------------------------------------- #
# Dataset constants
# --------------------------------------------------------------------------- #

#: Kaggle identifier of the dataset used in this project.  It is the APTOS 2019
#: Blindness Detection training set, resized to 224x224 by the dataset author.
KAGGLE_DATASET = "sovitrath/diabetic-retinopathy-224x224-2019-data"

#: Name of the zip file the Kaggle CLI produces for the dataset above.
KAGGLE_ZIP_NAME = "diabetic-retinopathy-224x224-2019-data.zip"

#: Canonical class names (index = severity grade used by the ICDR scale).
CLASS_NAMES: List[str] = ["No_DR", "Mild", "Moderate", "Severe", "Proliferative_DR"]

#: Human friendly labels used in figures and in the demo application.
CLASS_LABELS: List[str] = [
    "No DR",
    "Mild NPDR",
    "Moderate NPDR",
    "Severe NPDR",
    "Proliferative DR",
]

#: Short clinical description of each grade (International Clinical DR scale).
CLASS_DESCRIPTIONS: Dict[int, str] = {
    0: "No abnormalities visible.",
    1: "Microaneurysms only.",
    2: "More than microaneurysms but less than severe NPDR "
       "(haemorrhages, hard exudates, cotton-wool spots).",
    3: "Severe non-proliferative DR: extensive haemorrhages, venous beading "
       "or IRMA (4-2-1 rule).",
    4: "Proliferative DR: neovascularisation and/or vitreous or pre-retinal "
       "haemorrhage.",
}

#: Mapping from *every* folder / label spelling we have seen in the wild to the
#: numeric grade.  The dataset on Kaggle spells the last class
#: ``Proliferate_DR`` (sic), which is why several spellings are listed.
LABEL_TO_INDEX: Dict[str, int] = {
    "no_dr": 0, "no dr": 0, "nodr": 0, "0": 0,
    "mild": 1, "mild_dr": 1, "1": 1,
    "moderate": 2, "moderate_dr": 2, "2": 2,
    "severe": 3, "severe_dr": 3, "3": 3,
    "proliferative_dr": 4, "proliferate_dr": 4, "proliferative dr": 4,
    "proliferate dr": 4, "proliferative": 4, "proliferate": 4, "pdr": 4, "4": 4,
}

#: Grades >= this value are considered *referable* DR (moderate NPDR or worse),
#: the threshold used by most screening programmes and by Gulshan et al. (2016).
REFERABLE_THRESHOLD = 2

#: Supported preprocessing methods (see ``dr_detection.preprocessing``).
PREPROCESS_METHODS = ["raw", "ben_graham", "clahe", "clahe_ben_graham", "unsharp"]

#: Supported backbones (see ``dr_detection.model``).
BACKBONES = ["EfficientNetB0", "EfficientNetB3", "ResNet50V2", "MobileNetV2", "DenseNet121"]


def project_root() -> Path:
    """Return the repository root (two levels above this file)."""
    return Path(__file__).resolve().parents[2]


# --------------------------------------------------------------------------- #
# Run configuration
# --------------------------------------------------------------------------- #
@dataclass
class Config:
    """All hyper-parameters and paths for one experiment run.

    The defaults reproduce the *main* experiment reported in the coursework
    report: EfficientNetB0 + Ben Graham preprocessing, two-phase training.
    """

    # ---- paths -------------------------------------------------------------
    data_dir: str = str(project_root() / "data" / "diabetic_retinopathy_data")
    cache_dir: str = str(project_root() / "data" / "cache")
    results_dir: str = str(project_root() / "results")
    models_dir: str = str(project_root() / "models")

    # ---- images -------------------------------------------------------------
    img_size: int = 224
    channels: int = 3
    preprocess: str = "ben_graham"      # one of PREPROCESS_METHODS

    # ---- splits -------------------------------------------------------------
    val_frac: float = 0.10              # 70 / 10 / 20 split as in the lecturer's guide
    test_frac: float = 0.20
    seed: int = 42

    # ---- model --------------------------------------------------------------
    backbone: str = "EfficientNetB0"    # one of BACKBONES
    dropout: float = 0.3
    dense_units: tuple = (512, 256)

    # ---- training -----------------------------------------------------------
    batch_size: int = 32
    epochs_phase1: int = 10             # frozen backbone, train the head
    epochs_phase2: int = 20             # fine-tune the last `unfreeze_layers` layers
    lr_phase1: float = 1e-4
    lr_phase2: float = 1e-5
    unfreeze_layers: int = 30
    early_stopping_patience: int = 6
    reduce_lr_patience: int = 3
    use_class_weights: bool = True
    oversample_minority: bool = False   # alternative balancing strategy
    use_augmentation: bool = True

    # ---- misc ---------------------------------------------------------------
    run_name: str = ""                  # auto-generated when empty
    quick_test: bool = False            # tiny subset + 1 epoch, to smoke-test the pipeline
    quick_test_per_class: int = 24

    # ------------------------------------------------------------------------
    def __post_init__(self) -> None:
        if self.preprocess not in PREPROCESS_METHODS:
            raise ValueError(f"preprocess must be one of {PREPROCESS_METHODS}, got {self.preprocess!r}")
        if self.backbone not in BACKBONES:
            raise ValueError(f"backbone must be one of {BACKBONES}, got {self.backbone!r}")
        if not self.run_name:
            self.run_name = f"{self.backbone}_{self.preprocess}_{self.img_size}"
            if self.quick_test:
                self.run_name += "_quicktest"

    # convenience -------------------------------------------------------------
    @property
    def input_shape(self) -> tuple:
        return (self.img_size, self.img_size, self.channels)

    @property
    def num_classes(self) -> int:
        return len(CLASS_NAMES)

    @property
    def run_dir(self) -> Path:
        return Path(self.results_dir) / self.run_name

    @property
    def images_dir(self) -> Path:
        """Folder that contains one sub-folder per class."""
        for candidate in ("colored_images", "gaussian_filtered_images", "images", ""):
            p = Path(self.data_dir) / candidate if candidate else Path(self.data_dir)
            if p.exists() and any(c.is_dir() for c in p.iterdir()):
                return p
        return Path(self.data_dir) / "colored_images"

    def to_dict(self) -> dict:
        d = asdict(self)
        d["dense_units"] = list(self.dense_units)
        return d

    @classmethod
    def from_dict(cls, d: dict) -> "Config":
        d = dict(d)
        if "dense_units" in d:
            d["dense_units"] = tuple(d["dense_units"])
        return cls(**{k: v for k, v in d.items() if k in cls.__dataclass_fields__})

### `dr_detection/utils.py`
Seeding for reproducibility, JSON helpers, timing and hardware information.

In [ ]:
%%writefile src/dr_detection/utils.py
"""Small helpers: seeding, JSON I/O, timing and hardware info."""

from __future__ import annotations

import json
import os
import platform
import random
import time
from contextlib import contextmanager
from pathlib import Path
from typing import Any

import numpy as np


def set_seed(seed: int = 42) -> None:
    """Seed Python, NumPy and TensorFlow for reproducible experiments."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    try:
        import tensorflow as tf

        tf.random.set_seed(seed)
        try:
            from tensorflow import keras

            keras.utils.set_random_seed(seed)
        except Exception:
            pass
    except ImportError:
        pass


def ensure_dir(path: str | Path) -> Path:
    p = Path(path)
    p.mkdir(parents=True, exist_ok=True)
    return p


class _NumpyEncoder(json.JSONEncoder):
    """Make numpy scalars / arrays JSON serialisable."""

    def default(self, o: Any):
        if isinstance(o, (np.integer,)):
            return int(o)
        if isinstance(o, (np.floating,)):
            return float(o)
        if isinstance(o, np.ndarray):
            return o.tolist()
        if isinstance(o, Path):
            return str(o)
        return super().default(o)


def save_json(obj: Any, path: str | Path) -> None:
    ensure_dir(Path(path).parent)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, cls=_NumpyEncoder)


def load_json(path: str | Path) -> Any:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


@contextmanager
def timer(label: str = ""):
    """``with timer('training'): ...`` prints the elapsed time."""
    t0 = time.perf_counter()
    yield
    dt = time.perf_counter() - t0
    print(f"[timer] {label}: {dt/60:.1f} min ({dt:.0f} s)")


def hardware_info() -> dict:
    """Describe the machine the experiment ran on (goes into metrics.json)."""
    info = {"platform": platform.platform(), "python": platform.python_version()}
    try:
        import tensorflow as tf

        info["tensorflow"] = tf.__version__
        gpus = tf.config.list_physical_devices("GPU")
        info["gpu"] = [g.name for g in gpus]
        try:
            details = tf.config.experimental.get_device_details(gpus[0]) if gpus else {}
            info["gpu_name"] = details.get("device_name", "")
        except Exception:
            pass
    except ImportError:
        info["tensorflow"] = None
    return info

### `dr_detection/preprocessing.py`
Black-border crop, resize, Ben Graham, CLAHE, unsharp masking, edge map.

In [ ]:
%%writefile src/dr_detection/preprocessing.py
"""
Image preprocessing for colour fundus photographs.

Every function here takes and returns **RGB uint8 numpy arrays** so that the
same code can be used by the training pipeline (numpy / tf.data), by the
figure scripts and by the Streamlit demo application.

Pipeline (in the order applied by :func:`preprocess_image`):

1. **Black-border cropping** - fundus photographs contain a large black frame
   around the circular retina.  Cropping removes uninformative pixels and
   makes the retina fill the frame consistently.
2. **Resizing** to a square ``img_size`` so that all images share one shape.
3. **Contrast / edge enhancement** using one of

   * ``raw``              - nothing further (baseline)
   * ``ben_graham``       - Gaussian-blur subtraction (Graham, 2015).  Removes
                            illumination differences between cameras and
                            sharpens vessels, haemorrhages and exudates.
   * ``clahe``            - Contrast Limited Adaptive Histogram Equalisation on
                            the L channel of the LAB colour space.
   * ``clahe_ben_graham`` - CLAHE followed by Ben Graham.
   * ``unsharp``          - unsharp masking (edge enhancement).

Noise removal is achieved implicitly by the Gaussian component of Ben Graham
and by the median filter in :func:`denoise`.
"""

from __future__ import annotations

from typing import Dict

import cv2
import numpy as np

# --------------------------------------------------------------------------- #
# Loading
# --------------------------------------------------------------------------- #
def load_image(path: str) -> np.ndarray:
    """Read an image from disk and return it as **RGB uint8**.

    OpenCV loads images as BGR, so the channels are swapped here once and the
    rest of the code base can assume RGB everywhere.
    """
    img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(f"Could not read image: {path}")
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


# --------------------------------------------------------------------------- #
# Individual steps
# --------------------------------------------------------------------------- #
def crop_black_border(img: np.ndarray, tol: int = 7) -> np.ndarray:
    """Crop the black frame around the retina.

    A pixel is considered "background" when its grey level is <= ``tol``.
    Rows and columns that contain *only* background are removed.  If the mask
    is empty (completely black image) the image is returned unchanged.
    """
    if img.ndim == 2:
        gray = img
    else:
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    mask = gray > tol
    rows = np.where(mask.any(axis=1))[0]
    cols = np.where(mask.any(axis=0))[0]
    if rows.size == 0 or cols.size == 0:
        return img
    return img[rows[0]: rows[-1] + 1, cols[0]: cols[-1] + 1]


def resize(img: np.ndarray, size: int) -> np.ndarray:
    """Resize to ``size x size`` using area interpolation (good for shrinking)."""
    interp = cv2.INTER_AREA if img.shape[0] > size else cv2.INTER_CUBIC
    return cv2.resize(img, (size, size), interpolation=interp)


def ben_graham(img: np.ndarray, sigma: float = 10.0) -> np.ndarray:
    """Ben Graham's preprocessing (winner of the 2015 Kaggle DR competition).

    ``out = 4 * img - 4 * GaussianBlur(img) + 128``

    Subtracting a heavily blurred copy removes slow illumination gradients and
    colour casts, while the factor 4 amplifies local contrast, so that fine
    lesions (micro-aneurysms, haemorrhages) stand out.  The +128 offset keeps
    the result centred in the 0-255 range.
    """
    blurred = cv2.GaussianBlur(img, (0, 0), sigmaX=sigma)
    return cv2.addWeighted(img, 4, blurred, -4, 128)


def apply_clahe(img: np.ndarray, clip_limit: float = 2.0, tile_grid: int = 8) -> np.ndarray:
    """Contrast Limited Adaptive Histogram Equalisation.

    CLAHE is applied to the *lightness* channel only (LAB colour space) so that
    colours are preserved while local contrast is boosted.
    """
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l_channel, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(tile_grid, tile_grid))
    l_channel = clahe.apply(l_channel)
    lab = cv2.merge((l_channel, a, b))
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)


def unsharp_mask(img: np.ndarray, sigma: float = 3.0, amount: float = 1.5) -> np.ndarray:
    """Edge enhancement by unsharp masking: ``img + amount * (img - blur)``."""
    blurred = cv2.GaussianBlur(img, (0, 0), sigmaX=sigma)
    sharpened = cv2.addWeighted(img, 1 + amount, blurred, -amount, 0)
    return np.clip(sharpened, 0, 255).astype(np.uint8)


def denoise(img: np.ndarray, ksize: int = 3) -> np.ndarray:
    """Light median filtering to suppress sensor noise without blurring edges."""
    return cv2.medianBlur(img, ksize)


def circular_mask(img: np.ndarray, radius_frac: float = 0.98) -> np.ndarray:
    """Zero-out everything outside a centred circle.

    Ben Graham preprocessing turns the black frame grey; masking it back to
    black removes that artificial border so the network cannot use it.
    """
    h, w = img.shape[:2]
    mask = np.zeros((h, w), dtype=np.uint8)
    cv2.circle(mask, (w // 2, h // 2), int(min(h, w) // 2 * radius_frac), 255, -1)
    return cv2.bitwise_and(img, img, mask=mask)


def edge_map(img: np.ndarray, low: int = 50, high: int = 150) -> np.ndarray:
    """Canny edge map of the green channel (vessels are most visible in green).

    Used only for visualisation / EDA - it shows what "edge enhancement"
    actually emphasises in a fundus image.
    """
    green = img[:, :, 1] if img.ndim == 3 else img
    return cv2.Canny(green, low, high)


# --------------------------------------------------------------------------- #
# Full pipeline
# --------------------------------------------------------------------------- #
def preprocess_image(img: np.ndarray, method: str = "ben_graham", size: int = 224) -> np.ndarray:
    """Apply the complete preprocessing pipeline to one RGB uint8 image.

    Parameters
    ----------
    img    : RGB uint8 array of any size
    method : one of ``config.PREPROCESS_METHODS``
    size   : output side length in pixels

    Returns
    -------
    RGB uint8 array of shape ``(size, size, 3)``.
    """
    img = crop_black_border(img)
    img = resize(img, size)

    if method == "raw":
        return img
    if method == "ben_graham":
        return circular_mask(ben_graham(img))
    if method == "clahe":
        return apply_clahe(img)
    if method == "clahe_ben_graham":
        return circular_mask(ben_graham(apply_clahe(img)))
    if method == "unsharp":
        return unsharp_mask(img)
    raise ValueError(f"Unknown preprocessing method: {method!r}")


def preprocessing_stages(img: np.ndarray, size: int = 224) -> Dict[str, np.ndarray]:
    """Return every intermediate result, for the "before / after" figure."""
    cropped = crop_black_border(img)
    resized = resize(cropped, size)
    stages = {
        "1. Original": img,
        "2. Border crop": cropped,
        "3. Resize": resized,
        "4. CLAHE": apply_clahe(resized),
        "5. Ben Graham": circular_mask(ben_graham(resized)),
        "6. Unsharp mask": unsharp_mask(resized),
        "7. Edge map": edge_map(resized),
    }
    return stages

### `dr_detection/data.py`
Dataset discovery, robust label mapping, stratified split, cached preprocessing, tf.data pipeline.

In [ ]:
%%writefile src/dr_detection/data.py
"""
Dataset discovery, label mapping, stratified splitting and tf.data pipelines.

Design decisions
----------------
* The dataset is small enough (3,662 images at 224 px ~ 550 MB as uint8) to be
  **preprocessed once and kept in RAM** as numpy arrays.  This makes training
  I/O-free and guarantees the exact same preprocessing for every epoch.
* Preprocessed arrays are cached to ``cache_dir`` as ``.npz`` files keyed by
  preprocessing method and image size, so re-running an experiment is instant.
* Splitting is **stratified** on the label and seeded, so every experiment
  (different backbones / preprocessing) is evaluated on the *same* test images.
* The test split is created first and never touched again; validation is then
  carved out of the remaining training images (70 / 10 / 20 overall).
"""

from __future__ import annotations

import os
from pathlib import Path
from typing import Dict, Optional, Tuple

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

from . import config as C
from .preprocessing import load_image, preprocess_image

IMAGE_EXTENSIONS = (".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp")


# --------------------------------------------------------------------------- #
# Dataset discovery
# --------------------------------------------------------------------------- #
def _looks_like_class_folder_root(path: Path) -> bool:
    """True if ``path`` has at least 3 sub-folders whose names are DR grades."""
    try:
        subdirs = [p for p in path.iterdir() if p.is_dir()]
    except (PermissionError, OSError):
        return False
    return sum(normalise_label(p.name) is not None for p in subdirs) >= 3


def find_dataset_dir(search_root: str | Path = "/kaggle/input", max_depth: int = 5) -> Path:
    """Locate the dataset folder under ``search_root`` (e.g. the Kaggle input mount).

    Kaggle mounts attached datasets read-only below ``/kaggle/input`` but the exact
    sub-path depends on the dataset slug and on the Kaggle UI version, so instead
    of hard-coding it we walk the tree and return the folder that *contains*
    ``colored_images/`` (or, failing that, the parent of the class folders).
    The returned path is what ``Config.data_dir`` expects.
    """
    root = Path(search_root)
    if not root.exists():
        raise FileNotFoundError(f"{root} does not exist")
    root_depth = len(root.parts)
    for dirpath, dirnames, _ in os.walk(root):
        here = Path(dirpath)
        if len(here.parts) - root_depth > max_depth:
            dirnames[:] = []
            continue
        if "colored_images" in dirnames:
            return here
        if _looks_like_class_folder_root(here):
            return here.parent if here.name.lower() in ("colored_images", "gaussian_filtered_images", "images") else here
    raise FileNotFoundError(
        f"No diabetic retinopathy class folders found under {root}. "
        f"Attach the Kaggle dataset '{C.KAGGLE_DATASET}' with 'Add Input'."
    )


# --------------------------------------------------------------------------- #
# DataFrame creation
# --------------------------------------------------------------------------- #
def normalise_label(label: str) -> Optional[int]:
    """Map any label spelling (``'Proliferate_DR'``, ``'No DR'``, ``'3'`` ...) to 0-4."""
    key = str(label).strip().lower().replace("-", "_")
    if key in C.LABEL_TO_INDEX:
        return C.LABEL_TO_INDEX[key]
    key2 = key.replace("_", " ")
    if key2 in C.LABEL_TO_INDEX:
        return C.LABEL_TO_INDEX[key2]
    try:
        val = int(float(key))
        return val if 0 <= val <= 4 else None
    except ValueError:
        return None


def build_dataframe(images_dir: str | Path) -> pd.DataFrame:
    """Scan ``images_dir/<class_folder>/*.png`` into a tidy DataFrame.

    Columns: ``image_path``, ``filename``, ``id_code``, ``folder``,
    ``label`` (0-4) and ``class_name``.

    Unlike a naive ``{'No_DR': 0, ...}`` dictionary, :func:`normalise_label`
    understands the mis-spelt ``Proliferate_DR`` folder of the Kaggle dataset,
    so **no images are silently dropped**.
    """
    images_dir = Path(images_dir)
    if not images_dir.exists():
        raise FileNotFoundError(
            f"{images_dir} does not exist. Run scripts/download_data.py first."
        )
    rows = []
    for class_dir in sorted(p for p in images_dir.iterdir() if p.is_dir()):
        label = normalise_label(class_dir.name)
        if label is None:
            print(f"[data] WARNING: folder {class_dir.name!r} is not a known class - skipped")
            continue
        for f in sorted(class_dir.iterdir()):
            if f.suffix.lower() in IMAGE_EXTENSIONS:
                rows.append({
                    "image_path": str(f),
                    "filename": f.name,
                    "id_code": f.stem,
                    "folder": class_dir.name,
                    "label": label,
                    "class_name": C.CLASS_NAMES[label],
                })
    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError(f"No images found under {images_dir}")
    return df


def cross_check_with_csv(df: pd.DataFrame, data_dir: str | Path) -> pd.DataFrame:
    """If the dataset ships a ``train.csv`` (id_code, diagnosis) verify labels.

    Returns the DataFrame with an extra ``csv_diagnosis`` column and prints the
    number of disagreements (expected: 0).  This is a sanity check that the
    folder structure and the official label file agree.
    """
    csv_path = None
    for root, _, files in os.walk(data_dir):
        for f in files:
            if f.lower() == "train.csv":
                csv_path = Path(root) / f
                break
    if csv_path is None:
        return df
    csv = pd.read_csv(csv_path)
    if {"id_code", "diagnosis"} <= set(csv.columns):
        merged = df.merge(csv[["id_code", "diagnosis"]], on="id_code", how="left")
        merged = merged.rename(columns={"diagnosis": "csv_diagnosis"})
        mismatches = int((merged["csv_diagnosis"].notna() & (merged["csv_diagnosis"] != merged["label"])).sum())
        missing = int(merged["csv_diagnosis"].isna().sum())
        print(f"[data] train.csv cross-check: {mismatches} label mismatches, {missing} ids not in csv")
        return merged
    return df


def class_distribution(df: pd.DataFrame) -> pd.DataFrame:
    """Return a table with count and percentage per class (for the report)."""
    counts = df["label"].value_counts().sort_index()
    table = pd.DataFrame({
        "grade": counts.index,
        "class": [C.CLASS_LABELS[i] for i in counts.index],
        "images": counts.values,
        "percent": (100 * counts.values / len(df)).round(1),
    })
    return table.reset_index(drop=True)


# --------------------------------------------------------------------------- #
# Splitting
# --------------------------------------------------------------------------- #
def split_dataframe(
    df: pd.DataFrame,
    val_frac: float = 0.10,
    test_frac: float = 0.20,
    seed: int = 42,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Stratified train / validation / test split (defaults to 70 / 10 / 20)."""
    train_val, test = train_test_split(
        df, test_size=test_frac, stratify=df["label"], random_state=seed
    )
    # validation fraction relative to the remaining data
    rel_val = val_frac / (1.0 - test_frac)
    train, val = train_test_split(
        train_val, test_size=rel_val, stratify=train_val["label"], random_state=seed
    )
    for name, part in (("train", train), ("val", val), ("test", test)):
        part = part.copy()
        part["split"] = name
    return (
        train.reset_index(drop=True),
        val.reset_index(drop=True),
        test.reset_index(drop=True),
    )


def subsample_per_class(df: pd.DataFrame, n_per_class: int, seed: int = 42) -> pd.DataFrame:
    """Keep at most ``n_per_class`` images of every class (used by quick tests)."""
    parts = [
        g.sample(min(len(g), n_per_class), random_state=seed)
        for _, g in df.groupby("label")
    ]
    return pd.concat(parts).reset_index(drop=True)


# --------------------------------------------------------------------------- #
# Image loading with cache
# --------------------------------------------------------------------------- #
def load_images(
    df: pd.DataFrame,
    method: str,
    size: int,
    cache_dir: Optional[str | Path] = None,
    cache_tag: str = "",
    desc: str = "images",
) -> np.ndarray:
    """Load + preprocess every image of ``df`` into a ``uint8 (N, size, size, 3)`` array.

    If ``cache_dir`` is given the result is stored as ``.npz`` and re-used on the
    next call with the same ``method``, ``size``, ``cache_tag`` and number of
    rows (the cache also stores the id codes to detect a mismatch).
    """
    cache_path = None
    if cache_dir is not None:
        cache_dir = Path(cache_dir)
        cache_dir.mkdir(parents=True, exist_ok=True)
        cache_path = cache_dir / f"{cache_tag}_{method}_{size}_{len(df)}.npz"
        if cache_path.exists():
            data = np.load(cache_path, allow_pickle=False)
            if list(data["ids"]) == list(df["id_code"]):
                return data["X"]

    X = np.empty((len(df), size, size, 3), dtype=np.uint8)
    for i, path in enumerate(tqdm(df["image_path"], desc=f"Preprocessing {desc}", leave=False)):
        X[i] = preprocess_image(load_image(path), method=method, size=size)

    if cache_path is not None:
        np.savez(cache_path, X=X, ids=np.array(df["id_code"], dtype=str))
    return X


# --------------------------------------------------------------------------- #
# tf.data pipelines
# --------------------------------------------------------------------------- #
def make_tf_dataset(
    X: np.ndarray,
    y: np.ndarray,
    batch_size: int,
    shuffle: bool = False,
    augmentation=None,
    seed: int = 42,
):
    """Wrap numpy arrays into a batched, prefetched ``tf.data.Dataset``.

    ``augmentation`` is an optional Keras ``Sequential`` of preprocessing layers
    that is applied *per batch on the fly* (training set only).  Images are
    passed to the model as float32 in the 0-255 range; each backbone applies
    its own normalisation inside the model (see ``model.py``).
    """
    import tensorflow as tf  # imported lazily so figure scripts stay TF-free

    ds = tf.data.Dataset.from_tensor_slices((X, y.astype(np.int32)))
    if shuffle:
        ds = ds.shuffle(buffer_size=min(len(X), 4096), seed=seed, reshuffle_each_iteration=True)
    ds = ds.batch(batch_size)
    ds = ds.map(lambda img, lbl: (tf.cast(img, tf.float32), lbl), num_parallel_calls=tf.data.AUTOTUNE)
    if augmentation is not None:
        ds = ds.map(lambda img, lbl: (augmentation(img, training=True), lbl),
                    num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)


def prepare_data(cfg: C.Config, verbose: bool = True) -> Dict[str, object]:
    """One call that returns everything the training loop needs.

    Returns a dict with the three DataFrames (``df_train`` ...), the preprocessed
    arrays (``X_train`` ...) and label vectors (``y_train`` ...).
    """
    df = build_dataframe(cfg.images_dir)
    df = cross_check_with_csv(df, cfg.data_dir)
    if cfg.quick_test:
        df = subsample_per_class(df, cfg.quick_test_per_class, cfg.seed)
    df_train, df_val, df_test = split_dataframe(df, cfg.val_frac, cfg.test_frac, cfg.seed)

    if verbose:
        print(f"[data] total={len(df)}  train={len(df_train)}  val={len(df_val)}  test={len(df_test)}")

    tag = "quick" if cfg.quick_test else "full"
    out = {"df_all": df, "df_train": df_train, "df_val": df_val, "df_test": df_test}
    for name, part in (("train", df_train), ("val", df_val), ("test", df_test)):
        out[f"X_{name}"] = load_images(
            part, cfg.preprocess, cfg.img_size, cfg.cache_dir, cache_tag=f"{tag}_{name}", desc=name
        )
        out[f"y_{name}"] = part["label"].to_numpy()
    return out

### `dr_detection/augmentation.py`
On-the-fly GPU augmentation, inverse-frequency class weights, random over-sampling.

In [ ]:
%%writefile src/dr_detection/augmentation.py
"""
Data augmentation and class balancing.

Augmentation
------------
Fundus photographs are rotation- and flip-invariant (a rotated retina is still
the same retina) and vary in brightness / contrast between cameras.  We
therefore apply, *on the fly and only to the training set*:

* random horizontal + vertical flips
* random rotation (+/- 36 degrees)
* random zoom (+/- 10 %)
* random translation (+/- 5 %)
* random brightness (+/- 10 %)
* random contrast (+/- 10 %)

Because the augmentation is implemented with Keras preprocessing layers it runs
on the GPU and produces a *different* variant of every image at every epoch,
which is far more effective against over-fitting than a fixed, pre-generated
set of augmented copies.

Class balancing
---------------
The dataset is heavily imbalanced (49 % No DR vs 5 % Severe).  Two strategies
are provided:

* :func:`compute_class_weights` - inverse-frequency weights passed to
  ``model.fit(class_weight=...)`` (default; lecturer's recommended approach)
* :func:`oversample_minority`   - random over-sampling of the minority classes
  in the training split only (never in validation / test)
"""

from __future__ import annotations

from typing import Dict, Tuple

import numpy as np
from sklearn.utils.class_weight import compute_class_weight

from . import config as C


# --------------------------------------------------------------------------- #
# Augmentation pipeline
# --------------------------------------------------------------------------- #
def build_augmentation(seed: int = 42):
    """Return a Keras ``Sequential`` of random augmentation layers.

    The layers are only active when called with ``training=True`` (which
    ``data.make_tf_dataset`` does), so the *same* object can be shown in a
    model summary without affecting inference.
    """
    from tensorflow import keras
    from tensorflow.keras import layers

    return keras.Sequential(
        [
            layers.RandomFlip("horizontal_and_vertical", seed=seed),
            layers.RandomRotation(0.10, fill_mode="constant", fill_value=0.0, seed=seed),
            layers.RandomZoom(0.10, fill_mode="constant", fill_value=0.0, seed=seed),
            layers.RandomTranslation(0.05, 0.05, fill_mode="constant", fill_value=0.0, seed=seed),
            layers.RandomBrightness(0.10, value_range=(0.0, 255.0), seed=seed),
            layers.RandomContrast(0.10, seed=seed),
        ],
        name="augmentation",
    )


def augment_examples(img: np.ndarray, n: int = 8, seed: int = 0) -> np.ndarray:
    """Return ``n`` augmented variants of one uint8 image (for figures)."""
    import tensorflow as tf

    aug = build_augmentation(seed=seed)
    batch = tf.repeat(tf.cast(img[None, ...], tf.float32), n, axis=0)
    out = aug(batch, training=True).numpy()
    return np.clip(out, 0, 255).astype(np.uint8)


# --------------------------------------------------------------------------- #
# Class balancing
# --------------------------------------------------------------------------- #
def compute_class_weights(y: np.ndarray) -> Dict[int, float]:
    """Inverse-frequency ("balanced") class weights, as a ``{class: weight}`` dict.

    ``weight_c = N / (num_classes * n_c)``  - rare classes get a weight > 1,
    frequent classes a weight < 1, so every class contributes roughly equally
    to the loss.
    """
    classes = np.arange(C.Config().num_classes)
    present = np.unique(y)
    weights = compute_class_weight(class_weight="balanced", classes=present, y=y)
    out = {int(c): 1.0 for c in classes}
    out.update({int(c): float(w) for c, w in zip(present, weights)})
    return out


def oversample_minority(X: np.ndarray, y: np.ndarray, seed: int = 42) -> Tuple[np.ndarray, np.ndarray]:
    """Random over-sampling so that every class has as many samples as the largest.

    Only ever apply this to the **training** split.  Because augmentation is
    applied on the fly, duplicated images still receive different random
    transformations, so this is *not* simple duplication of pixels.
    """
    rng = np.random.default_rng(seed)
    classes, counts = np.unique(y, return_counts=True)
    target = counts.max()
    idx = []
    for c, n in zip(classes, counts):
        cls_idx = np.where(y == c)[0]
        extra = rng.choice(cls_idx, size=target - n, replace=True) if n < target else np.array([], dtype=int)
        idx.append(np.concatenate([cls_idx, extra]))
    idx = np.concatenate(idx)
    rng.shuffle(idx)
    return X[idx], y[idx]


def balance_summary(y_before: np.ndarray, y_after: np.ndarray | None, weights: Dict[int, float]):
    """Small table (list of dicts) describing the balancing applied - for the report."""
    rows = []
    for c in range(len(C.CLASS_NAMES)):
        rows.append({
            "grade": c,
            "class": C.CLASS_LABELS[c],
            "train_images": int((y_before == c).sum()),
            "after_oversampling": int((y_after == c).sum()) if y_after is not None else int((y_before == c).sum()),
            "class_weight": round(weights.get(c, 1.0), 3),
        })
    return rows

### `dr_detection/model.py`
ImageNet backbones with the transfer-learning head and fine-tuning helpers.

In [ ]:
%%writefile src/dr_detection/model.py
"""
CNN architectures with transfer learning.

Architecture (identical for every backbone)::

    Input (224, 224, 3)  - float32, 0-255
      |
    Backbone-specific normalisation  (built into the graph so the saved model
      |                               accepts raw images)
    ImageNet-pretrained backbone, include_top=False   (frozen in phase 1)
      |
    GlobalAveragePooling2D
      |
    Dropout(p) -> Dense(512, ReLU) -> Dropout(p) -> Dense(256, ReLU)
      |
    Dense(5, softmax)      -> probability of each DR grade

Why these backbones?
--------------------
* **EfficientNetB0 / B3** - best accuracy per FLOP; B0 trains quickly on a
  free Colab GPU and was designed with compound scaling (Tan & Le, 2019).
* **ResNet50V2**  - the classic residual network, strong baseline.
* **MobileNetV2** - depth-wise separable convolutions; tiny and fast, ideal
  for on-device screening in low-resource clinics.
* **DenseNet121** - dense connectivity, popular in medical imaging papers.

Transfer learning is applied in two phases (see ``train.py``): first only the
new head is trained with the backbone frozen, then the top ``unfreeze_layers``
layers of the backbone are un-frozen and trained with a 10x smaller learning
rate.  BatchNormalization layers stay frozen during fine-tuning - updating
their running statistics with small batches of a new domain is a well known
cause of accuracy collapse when fine-tuning EfficientNet.
"""

from __future__ import annotations

from typing import Tuple

from . import config as C


def _normalisation_layer(backbone: str):
    """Return the input normalisation layer expected by ``backbone``.

    * EfficientNet models in Keras already contain their own rescaling and
      normalisation layers, so the identity is returned.
    * ResNet50V2 / MobileNetV2 use "tf" mode: ``x / 127.5 - 1``  (-1 ... 1)
    * DenseNet121 uses "torch" mode: ``(x / 255 - mean) / std`` per channel
    """
    from tensorflow.keras import layers

    if backbone.startswith("EfficientNet"):
        return layers.Identity(name="normalisation_identity")
    if backbone in ("ResNet50V2", "MobileNetV2"):
        return layers.Rescaling(scale=1.0 / 127.5, offset=-1.0, name="normalisation_tf")
    if backbone == "DenseNet121":
        mean = [0.485, 0.456, 0.406]
        std = [0.229, 0.224, 0.225]
        scale = [1.0 / (255.0 * s) for s in std]
        offset = [-m / s for m, s in zip(mean, std)]
        return layers.Rescaling(scale=scale, offset=offset, name="normalisation_torch")
    raise ValueError(f"Unknown backbone {backbone!r}")


def _backbone_constructor(backbone: str):
    from tensorflow.keras import applications as A

    return {
        "EfficientNetB0": A.EfficientNetB0,
        "EfficientNetB3": A.EfficientNetB3,
        "ResNet50V2": A.ResNet50V2,
        "MobileNetV2": A.MobileNetV2,
        "DenseNet121": A.DenseNet121,
    }[backbone]


def build_model(cfg: C.Config, weights: str | None = "imagenet"):
    """Create the transfer-learning model described in the module docstring.

    Returns
    -------
    model : ``keras.Model`` - the full classifier (raw 0-255 images in)
    base  : ``keras.Model`` - the pretrained backbone (a layer inside ``model``)
    """
    from tensorflow import keras
    from tensorflow.keras import layers

    constructor = _backbone_constructor(cfg.backbone)
    base = constructor(weights=weights, include_top=False, input_shape=cfg.input_shape)
    base._name = "backbone"  # stable name so the app / Grad-CAM can find it
    base.trainable = False   # phase 1: frozen feature extractor

    inputs = keras.Input(shape=cfg.input_shape, name="fundus_image")
    x = _normalisation_layer(cfg.backbone)(inputs)
    x = base(x, training=False)  # training=False keeps BatchNorm in inference mode
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dropout(cfg.dropout, name="dropout_1")(x)
    x = layers.Dense(cfg.dense_units[0], activation="relu", name="fc_1")(x)
    x = layers.Dropout(cfg.dropout, name="dropout_2")(x)
    x = layers.Dense(cfg.dense_units[1], activation="relu", name="fc_2")(x)
    outputs = layers.Dense(cfg.num_classes, activation="softmax", name="dr_grade")(x)

    model = keras.Model(inputs, outputs, name=f"DR_{cfg.backbone}")
    return model, base


def get_backbone(model):
    """Return the nested backbone model of a classifier built by :func:`build_model`."""
    from tensorflow import keras

    for layer in model.layers:
        if isinstance(layer, keras.Model):
            return layer
    raise ValueError("No nested backbone model found")


def set_fine_tuning(base, unfreeze_layers: int) -> Tuple[int, int]:
    """Un-freeze the top ``unfreeze_layers`` layers of ``base`` (phase 2).

    BatchNormalization layers are kept frozen.  Returns the number of
    trainable and frozen layers for logging.
    """
    from tensorflow.keras import layers

    base.trainable = True
    n_layers = len(base.layers)
    cutoff = max(0, n_layers - unfreeze_layers)
    trainable = 0
    for i, layer in enumerate(base.layers):
        if i < cutoff or isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
        else:
            layer.trainable = True
            trainable += 1
    return trainable, n_layers - trainable


def count_params(model) -> dict:
    """Trainable / non-trainable / total parameter counts as plain ints."""
    import numpy as np

    trainable = int(sum(np.prod(w.shape) for w in model.trainable_weights))
    non_trainable = int(sum(np.prod(w.shape) for w in model.non_trainable_weights))
    return {"trainable": trainable, "non_trainable": non_trainable, "total": trainable + non_trainable}


def compile_model(model, learning_rate: float):
    """Compile with Adam + sparse categorical cross-entropy (integer labels)."""
    from tensorflow import keras

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

### `dr_detection/evaluate.py`
Accuracy, precision/recall/F1, QWK, referable-DR metrics and every evaluation figure.

In [ ]:
%%writefile src/dr_detection/evaluate.py
"""
Model evaluation: metrics, curves, confusion matrix and error analysis.

Metrics reported
----------------
* accuracy
* precision / recall / F1 - per class, macro and weighted average
* quadratic weighted kappa (QWK) - the official APTOS 2019 metric; it rewards
  predictions that are *close* to the true grade and penalises far misses,
  which matches the ordinal nature of DR staging
* **referable-DR** sensitivity / specificity / AUC - the clinically relevant
  binary question "should this patient be referred to an ophthalmologist?"
  (grade >= 2, i.e. moderate NPDR or worse)
* **any-DR** sensitivity / specificity - grade >= 1
* one-vs-rest ROC curves and AUC per class

All figures are written as PNG at 150 dpi so they can be pasted directly into
the report.
"""

from __future__ import annotations

from pathlib import Path
from typing import Dict, Optional

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import seaborn as sns  # noqa: E402
from sklearn.metrics import (  # noqa: E402
    accuracy_score,
    classification_report,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
    roc_auc_score,
    roc_curve,
)

from . import config as C  # noqa: E402

sns.set_theme(style="whitegrid", context="paper", font_scale=1.1)
PALETTE = ["#2a9d8f", "#e9c46a", "#f4a261", "#e76f51", "#9b2226"]


# --------------------------------------------------------------------------- #
# Predictions
# --------------------------------------------------------------------------- #
def predict_probs(model, X: np.ndarray, batch_size: int = 32) -> np.ndarray:
    """Softmax probabilities for a uint8 image array (N, H, W, 3)."""
    return model.predict(X.astype(np.float32), batch_size=batch_size, verbose=0)


def _binary_stats(y_true_bin: np.ndarray, y_pred_bin: np.ndarray, prob: Optional[np.ndarray]) -> Dict[str, float]:
    tp = int(((y_true_bin == 1) & (y_pred_bin == 1)).sum())
    tn = int(((y_true_bin == 0) & (y_pred_bin == 0)).sum())
    fp = int(((y_true_bin == 0) & (y_pred_bin == 1)).sum())
    fn = int(((y_true_bin == 1) & (y_pred_bin == 0)).sum())
    out = {
        "sensitivity": tp / max(tp + fn, 1),
        "specificity": tn / max(tn + fp, 1),
        "ppv": tp / max(tp + fp, 1),
        "npv": tn / max(tn + fn, 1),
        "tp": tp, "tn": tn, "fp": fp, "fn": fn,
    }
    if prob is not None and len(np.unique(y_true_bin)) == 2:
        out["auc"] = float(roc_auc_score(y_true_bin, prob))
    return out


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray, y_prob: Optional[np.ndarray] = None) -> dict:
    """All scalar metrics in one JSON-friendly dictionary."""
    labels = list(range(len(C.CLASS_NAMES)))
    p, r, f, s = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)
    metrics = {
        "n_test": int(len(y_true)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_precision": float(p.mean()),
        "macro_recall": float(r.mean()),
        "macro_f1": float(f.mean()),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "quadratic_weighted_kappa": float(cohen_kappa_score(y_true, y_pred, weights="quadratic")),
        "cohen_kappa": float(cohen_kappa_score(y_true, y_pred)),
        "per_class": {
            C.CLASS_NAMES[i]: {"precision": float(p[i]), "recall": float(r[i]),
                               "f1": float(f[i]), "support": int(s[i])}
            for i in labels
        },
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=labels).tolist(),
    }
    # clinically meaningful binary views ---------------------------------
    ref_prob = y_prob[:, C.REFERABLE_THRESHOLD:].sum(axis=1) if y_prob is not None else None
    any_prob = y_prob[:, 1:].sum(axis=1) if y_prob is not None else None
    ref = _binary_stats((y_true >= C.REFERABLE_THRESHOLD).astype(int),
                        (y_pred >= C.REFERABLE_THRESHOLD).astype(int), ref_prob)
    any_dr = _binary_stats((y_true >= 1).astype(int), (y_pred >= 1).astype(int), any_prob)
    metrics["referable_dr"] = ref
    metrics["any_dr"] = any_dr
    metrics["referable_sensitivity"] = ref["sensitivity"]
    metrics["referable_specificity"] = ref["specificity"]
    if y_prob is not None:
        try:
            metrics["macro_auc_ovr"] = float(roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro"))
        except ValueError:
            pass
    # mean absolute grade error - how far off are we on the ordinal scale?
    metrics["mean_absolute_grade_error"] = float(np.abs(y_true - y_pred).mean())
    metrics["within_one_grade_accuracy"] = float((np.abs(y_true - y_pred) <= 1).mean())
    return metrics


# --------------------------------------------------------------------------- #
# Figures
# --------------------------------------------------------------------------- #
def plot_training_curves(hist: pd.DataFrame, path: Path, title: str = "") -> None:
    """Accuracy and loss vs. epoch, with the phase-1 / phase-2 boundary marked."""
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    boundary = int((hist["phase"] == 1).sum()) + 0.5 if "phase" in hist else None
    for ax, key, label in ((axes[0], "accuracy", "Accuracy"), (axes[1], "loss", "Loss")):
        ax.plot(hist["epoch"], hist[key], marker="o", ms=3, label=f"train {label.lower()}", color="#264653")
        ax.plot(hist["epoch"], hist[f"val_{key}"], marker="s", ms=3, label=f"validation {label.lower()}", color="#e76f51")
        if boundary and boundary < hist["epoch"].max():
            ax.axvline(boundary, ls="--", color="grey", lw=1)
            ax.text(boundary + 0.2, ax.get_ylim()[1] * 0.98, "fine-tuning starts", va="top", fontsize=8, color="grey")
        ax.set_xlabel("Epoch")
        ax.set_ylabel(label)
        ax.set_title(f"{label} curve")
        ax.legend()
    if title:
        fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)


def plot_confusion_matrix(cm: np.ndarray, path: Path, normalize: bool = False, title: str = "") -> None:
    cm = np.asarray(cm)
    if normalize:
        cm_disp = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
        fmt, vmax = ".2f", 1.0
    else:
        cm_disp, fmt, vmax = cm, "d", None
    fig, ax = plt.subplots(figsize=(6.5, 5.5))
    sns.heatmap(cm_disp, annot=True, fmt=fmt, cmap="Blues", vmax=vmax, cbar=True,
                xticklabels=C.CLASS_LABELS, yticklabels=C.CLASS_LABELS, ax=ax)
    ax.set_xlabel("Predicted grade")
    ax.set_ylabel("True grade")
    ax.set_title(title or ("Normalised confusion matrix" if normalize else "Confusion matrix"))
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)


def plot_per_class_metrics(metrics: dict, path: Path) -> None:
    """Grouped bar chart of precision / recall / F1 per class."""
    pc = metrics["per_class"]
    df = pd.DataFrame({
        "class": [C.CLASS_LABELS[C.CLASS_NAMES.index(k)] for k in pc],
        "precision": [v["precision"] for v in pc.values()],
        "recall": [v["recall"] for v in pc.values()],
        "f1": [v["f1"] for v in pc.values()],
    }).melt(id_vars="class", var_name="metric", value_name="score")
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.barplot(data=df, x="class", y="score", hue="metric", ax=ax, palette=["#264653", "#2a9d8f", "#e9c46a"])
    ax.set_ylim(0, 1.05)
    ax.set_xlabel("")
    ax.set_ylabel("Score")
    ax.set_title("Per-class precision, recall and F1-score (test set)")
    for container in ax.containers:
        ax.bar_label(container, fmt="%.2f", fontsize=7, padding=1)
    ax.legend(loc="lower right")
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)


def plot_roc_curves(y_true: np.ndarray, y_prob: np.ndarray, path: Path) -> Dict[str, float]:
    """One-vs-rest ROC curve per class plus the referable-DR curve."""
    fig, ax = plt.subplots(figsize=(6, 5.5))
    aucs = {}
    for i, name in enumerate(C.CLASS_LABELS):
        y_bin = (y_true == i).astype(int)
        if y_bin.sum() == 0 or y_bin.sum() == len(y_bin):
            continue
        fpr, tpr, _ = roc_curve(y_bin, y_prob[:, i])
        auc = roc_auc_score(y_bin, y_prob[:, i])
        aucs[name] = float(auc)
        ax.plot(fpr, tpr, color=PALETTE[i], lw=1.6, label=f"{name} (AUC = {auc:.3f})")
    y_ref = (y_true >= C.REFERABLE_THRESHOLD).astype(int)
    if 0 < y_ref.sum() < len(y_ref):
        p_ref = y_prob[:, C.REFERABLE_THRESHOLD:].sum(axis=1)
        fpr, tpr, _ = roc_curve(y_ref, p_ref)
        auc = roc_auc_score(y_ref, p_ref)
        aucs["Referable DR"] = float(auc)
        ax.plot(fpr, tpr, color="black", lw=2.2, ls="--", label=f"Referable DR (AUC = {auc:.3f})")
    ax.plot([0, 1], [0, 1], color="lightgrey", ls=":")
    ax.set_xlabel("False positive rate (1 - specificity)")
    ax.set_ylabel("True positive rate (sensitivity)")
    ax.set_title("ROC curves (one-vs-rest)")
    ax.legend(fontsize=8, loc="lower right")
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return aucs


def plot_misclassified(X: np.ndarray, y_true: np.ndarray, y_pred: np.ndarray, y_prob: np.ndarray,
                       path: Path, n: int = 12) -> None:
    """Show the ``n`` most confident *wrong* predictions for error analysis."""
    wrong = np.where(y_true != y_pred)[0]
    if wrong.size == 0:
        return
    conf = y_prob[wrong, y_pred[wrong]]
    order = wrong[np.argsort(-conf)][:n]
    cols = 6
    rows = int(np.ceil(len(order) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(2.6 * cols, 2.9 * rows))
    axes = np.atleast_1d(axes).ravel()
    for ax in axes:
        ax.axis("off")
    for ax, i in zip(axes, order):
        ax.imshow(X[i])
        ax.set_title(f"true: {C.CLASS_LABELS[y_true[i]]}\npred: {C.CLASS_LABELS[y_pred[i]]} ({y_prob[i, y_pred[i]]:.2f})",
                     fontsize=8, color="#9b2226" if abs(y_true[i] - y_pred[i]) > 1 else "black")
    fig.suptitle("Most confident misclassifications (red = off by more than one grade)")
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)


def plot_prediction_distribution(y_true: np.ndarray, y_pred: np.ndarray, path: Path) -> None:
    """True vs predicted class counts - shows whether the model is biased to the majority class."""
    df = pd.DataFrame({
        "class": C.CLASS_LABELS * 2,
        "count": [int((y_true == i).sum()) for i in range(5)] + [int((y_pred == i).sum()) for i in range(5)],
        "which": ["true"] * 5 + ["predicted"] * 5,
    })
    fig, ax = plt.subplots(figsize=(7, 3.8))
    sns.barplot(data=df, x="class", y="count", hue="which", ax=ax, palette=["#264653", "#e9c46a"])
    ax.set_xlabel("")
    ax.set_title("True vs predicted grade distribution (test set)")
    for container in ax.containers:
        ax.bar_label(container, fontsize=8)
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)


# --------------------------------------------------------------------------- #
# Orchestration
# --------------------------------------------------------------------------- #
def evaluate_model(model, X_test: np.ndarray, y_test: np.ndarray, run_dir: Path,
                   batch_size: int = 32, make_figures: bool = True, prefix: str = "") -> dict:
    """Predict on the test set, compute every metric and write figures + files."""
    run_dir = Path(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)
    y_prob = predict_probs(model, X_test, batch_size)
    y_pred = y_prob.argmax(axis=1)

    metrics = compute_metrics(y_test, y_pred, y_prob)
    report = classification_report(y_test, y_pred, labels=list(range(5)),
                                   target_names=C.CLASS_LABELS, zero_division=0, digits=4)
    metrics["classification_report"] = report
    with open(run_dir / f"{prefix}classification_report.txt", "w", encoding="utf-8") as f:
        f.write(report)
    pd.DataFrame({"y_true": y_test, "y_pred": y_pred,
                  **{f"p_{C.CLASS_NAMES[i]}": y_prob[:, i] for i in range(5)}}
                 ).to_csv(run_dir / f"{prefix}test_predictions.csv", index=False)

    if make_figures:
        cm = np.array(metrics["confusion_matrix"])
        plot_confusion_matrix(cm, run_dir / f"{prefix}confusion_matrix.png")
        plot_confusion_matrix(cm, run_dir / f"{prefix}confusion_matrix_normalised.png", normalize=True)
        plot_per_class_metrics(metrics, run_dir / f"{prefix}per_class_metrics.png")
        metrics["roc_auc"] = plot_roc_curves(y_test, y_prob, run_dir / f"{prefix}roc_curves.png")
        plot_misclassified(X_test, y_test, y_pred, y_prob, run_dir / f"{prefix}misclassified.png")
        plot_prediction_distribution(y_test, y_pred, run_dir / f"{prefix}prediction_distribution.png")
    print(report)
    return metrics

### `dr_detection/gradcam.py`
Grad-CAM explainability heat-maps.

In [ ]:
%%writefile src/dr_detection/gradcam.py
"""
Grad-CAM (Gradient-weighted Class Activation Mapping) explainability.

Grad-CAM (Selvaraju et al., 2017) answers the question *"which part of the
retina made the network choose this grade?"*.  It back-propagates the score of
the predicted class to the last convolutional feature map of the backbone,
averages the gradients per channel to obtain importance weights, and builds a
weighted sum of the feature maps.  High values mark the regions that increased
the class score - ideally lesions such as haemorrhages or exudates rather than
the image border.

Implementation note
-------------------
Our classifier is a functional model of the form
``input -> normalisation -> backbone (nested model) -> head layers``.  Because
``include_top=False`` makes the backbone's *output* equal to its last
convolutional activation, we can split the forward pass into
``pre -> backbone -> post`` and take gradients w.r.t. the backbone output.  This
avoids the "graph disconnected" error that occurs when trying to build a
sub-model from a layer inside a nested Keras 3 model.
"""

from __future__ import annotations

from pathlib import Path
from typing import Optional, Tuple

import cv2
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402

from . import config as C  # noqa: E402
from .model import get_backbone  # noqa: E402


def make_gradcam_heatmap(model, img: np.ndarray, pred_index: Optional[int] = None) -> Tuple[np.ndarray, np.ndarray, int]:
    """Compute the Grad-CAM heat-map for one image.

    Parameters
    ----------
    model      : classifier built by ``model.build_model`` (or loaded from disk)
    img        : uint8 / float RGB image ``(H, W, 3)`` already preprocessed
    pred_index : class to explain; defaults to the predicted class

    Returns
    -------
    heatmap : float32 array in 0-1 with the backbone's spatial resolution (e.g. 7x7)
    probs   : the softmax probabilities of the image
    index   : the class index that was explained
    """
    import tensorflow as tf

    base = get_backbone(model)
    layers_list = list(model.layers)
    idx = layers_list.index(base)
    pre_layers = [l for l in layers_list[1:idx]]     # skip InputLayer
    post_layers = layers_list[idx + 1:]

    x = tf.convert_to_tensor(img[None, ...].astype(np.float32))
    for layer in pre_layers:
        x = layer(x)

    with tf.GradientTape() as tape:
        conv = base(x, training=False)
        tape.watch(conv)
        h = conv
        for layer in post_layers:
            h = layer(h)
        probs = h
        if pred_index is None:
            pred_index = int(tf.argmax(probs[0]))
        score = probs[:, pred_index]

    grads = tape.gradient(score, conv)                     # (1, h, w, c)
    pooled = tf.reduce_mean(grads, axis=(0, 1, 2))         # (c,)
    heatmap = tf.reduce_sum(conv[0] * pooled, axis=-1)     # (h, w)
    heatmap = tf.nn.relu(heatmap)
    heatmap = heatmap / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy().astype(np.float32), probs.numpy()[0], int(pred_index)


def overlay_heatmap(img: np.ndarray, heatmap: np.ndarray, alpha: float = 0.4) -> np.ndarray:
    """Blend a JET-coloured heat-map over the RGB image (returns uint8 RGB)."""
    h, w = img.shape[:2]
    hm = cv2.resize(heatmap, (w, h), interpolation=cv2.INTER_CUBIC)
    hm = np.uint8(255 * np.clip(hm, 0, 1))
    coloured = cv2.applyColorMap(hm, cv2.COLORMAP_JET)
    coloured = cv2.cvtColor(coloured, cv2.COLOR_BGR2RGB)
    out = cv2.addWeighted(img.astype(np.uint8), 1 - alpha, coloured, alpha, 0)
    return out


def gradcam_grid(model, X: np.ndarray, y: np.ndarray, path: Path, n_per_class: int = 2, seed: int = 0) -> None:
    """Figure: for each class, ``n_per_class`` test images with their Grad-CAM overlay."""
    rng = np.random.default_rng(seed)
    n_classes = len(C.CLASS_NAMES)
    fig, axes = plt.subplots(n_classes, 2 * n_per_class, figsize=(2.6 * 2 * n_per_class, 2.6 * n_classes))
    for c in range(n_classes):
        idx = np.where(y == c)[0]
        pick = rng.choice(idx, size=min(n_per_class, len(idx)), replace=False) if len(idx) else []
        for j in range(n_per_class):
            ax_img, ax_cam = axes[c, 2 * j], axes[c, 2 * j + 1]
            ax_img.axis("off")
            ax_cam.axis("off")
            if j >= len(pick):
                continue
            i = pick[j]
            heatmap, probs, k = make_gradcam_heatmap(model, X[i])
            ax_img.imshow(X[i])
            ax_img.set_title(f"true: {C.CLASS_LABELS[c]}", fontsize=8)
            ax_cam.imshow(overlay_heatmap(X[i], heatmap))
            ax_cam.set_title(f"pred: {C.CLASS_LABELS[k]} ({probs[k]:.2f})", fontsize=8,
                             color="black" if k == c else "#9b2226")
    fig.suptitle("Grad-CAM: regions that drove the prediction (red = most important)")
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)

### `dr_detection/visualize.py`
Exploratory data analysis, preprocessing and augmentation figures.

In [ ]:
%%writefile src/dr_detection/visualize.py
"""
Exploratory data analysis and pipeline figures (CPU only, no TensorFlow needed
except for the augmentation grid).

Every function writes one PNG and is called by ``scripts/make_figures.py`` and
by the Colab notebook, so the report figures are fully reproducible.
"""

from __future__ import annotations

from pathlib import Path
from typing import Dict, Optional

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import seaborn as sns  # noqa: E402

from . import config as C  # noqa: E402
from .preprocessing import load_image, preprocess_image, preprocessing_stages  # noqa: E402

sns.set_theme(style="whitegrid", context="paper", font_scale=1.1)
PALETTE = ["#2a9d8f", "#e9c46a", "#f4a261", "#e76f51", "#9b2226"]


def plot_class_distribution(df: pd.DataFrame, path: Path, title: str = "Class distribution (all images)") -> None:
    counts = df["label"].value_counts().sort_index()
    fig, ax = plt.subplots(figsize=(7.5, 4))
    bars = ax.bar([C.CLASS_LABELS[i] for i in counts.index], counts.values, color=PALETTE)
    for b, v in zip(bars, counts.values):
        ax.text(b.get_x() + b.get_width() / 2, v + 15, f"{v}\n({100*v/len(df):.1f}%)", ha="center", fontsize=9)
    ax.set_ylabel("Number of images")
    ax.set_title(title)
    ax.set_ylim(0, counts.max() * 1.18)
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)


def plot_split_distribution(df_train: pd.DataFrame, df_val: pd.DataFrame, df_test: pd.DataFrame, path: Path) -> None:
    """Stacked bars proving that the stratified split preserved class ratios."""
    rows = []
    for name, part in (("train", df_train), ("validation", df_val), ("test", df_test)):
        for c in range(5):
            rows.append({"split": name, "class": C.CLASS_LABELS[c], "images": int((part["label"] == c).sum())})
    d = pd.DataFrame(rows)
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.barplot(data=d, x="class", y="images", hue="split", ax=ax, palette=["#264653", "#2a9d8f", "#e9c46a"])
    for container in ax.containers:
        ax.bar_label(container, fontsize=7)
    ax.set_xlabel("")
    ax.set_title("Stratified 70 / 10 / 20 split per class")
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)


def plot_sample_grid(df: pd.DataFrame, path: Path, n_per_class: int = 4, method: str = "raw",
                     size: int = 224, seed: int = 0) -> None:
    """``n_per_class`` random images for each grade (raw or preprocessed)."""
    fig, axes = plt.subplots(5, n_per_class, figsize=(2.4 * n_per_class, 2.5 * 5))
    for c in range(5):
        sample = df[df["label"] == c].sample(n_per_class, random_state=seed)
        for j, (_, row) in enumerate(sample.iterrows()):
            ax = axes[c, j]
            ax.imshow(preprocess_image(load_image(row["image_path"]), method=method, size=size))
            ax.axis("off")
            if j == 0:
                ax.set_title(f"Grade {c}: {C.CLASS_LABELS[c]}", fontsize=9, loc="left")
    fig.suptitle("Sample fundus images per DR grade" + ("" if method == "raw" else f" ({method})"))
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)


def plot_preprocessing_stages(image_path: str, path: Path, size: int = 224) -> None:
    """One image through every preprocessing stage (before / after evidence)."""
    stages = preprocessing_stages(load_image(image_path), size=size)
    fig, axes = plt.subplots(1, len(stages), figsize=(2.6 * len(stages), 3))
    for ax, (name, img) in zip(axes, stages.items()):
        ax.imshow(img, cmap="gray" if img.ndim == 2 else None)
        ax.set_title(name, fontsize=9)
        ax.axis("off")
    fig.suptitle("Preprocessing pipeline stages")
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)


def plot_preprocessing_comparison(df: pd.DataFrame, path: Path, size: int = 224, seed: int = 0,
                                  methods=("raw", "clahe", "ben_graham", "clahe_ben_graham")) -> None:
    """One image per grade, shown under each preprocessing method."""
    fig, axes = plt.subplots(5, len(methods), figsize=(2.6 * len(methods), 2.6 * 5))
    for c in range(5):
        row = df[df["label"] == c].sample(1, random_state=seed).iloc[0]
        img = load_image(row["image_path"])
        for j, m in enumerate(methods):
            ax = axes[c, j]
            ax.imshow(preprocess_image(img, method=m, size=size))
            ax.axis("off")
            if c == 0:
                ax.set_title(m, fontsize=10)
            if j == 0:
                ax.text(-0.08, 0.5, C.CLASS_LABELS[c], transform=ax.transAxes, rotation=90,
                        va="center", ha="right", fontsize=9)
    fig.suptitle("Effect of each preprocessing method")
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)


def plot_intensity_stats(df: pd.DataFrame, path: Path, n_per_class: int = 60, seed: int = 0) -> pd.DataFrame:
    """Mean brightness and contrast per class - shows camera / illumination variability.

    Returns the table that the plot was made from (also useful in the report).
    """
    rows = []
    for c in range(5):
        sample = df[df["label"] == c].sample(min(n_per_class, int((df["label"] == c).sum())), random_state=seed)
        for _, r in sample.iterrows():
            img = load_image(r["image_path"])
            gray = img.mean(axis=2)
            fg = gray[gray > 7]  # ignore black frame
            rows.append({"class": C.CLASS_LABELS[c], "mean_brightness": fg.mean(), "contrast_std": fg.std(),
                         "height": img.shape[0], "width": img.shape[1]})
    d = pd.DataFrame(rows)
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
    sns.boxplot(data=d, x="class", y="mean_brightness", ax=axes[0], palette=PALETTE, hue="class", legend=False)
    axes[0].set_title("Mean brightness of retina pixels")
    axes[0].set_xlabel("")
    sns.boxplot(data=d, x="class", y="contrast_std", ax=axes[1], palette=PALETTE, hue="class", legend=False)
    axes[1].set_title("Contrast (std of retina pixels)")
    axes[1].set_xlabel("")
    for ax in axes:
        plt.setp(ax.get_xticklabels(), rotation=20, ha="right")
    fig.suptitle("Illumination variability across the dataset (raw images)")
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return d


def plot_augmentation_grid(image_path: str, path: Path, method: str = "ben_graham", size: int = 224, n: int = 7) -> None:
    """Original preprocessed image followed by ``n`` random augmentations."""
    from .augmentation import augment_examples

    img = preprocess_image(load_image(image_path), method=method, size=size)
    variants = augment_examples(img, n=n, seed=1)
    fig, axes = plt.subplots(1, n + 1, figsize=(2.4 * (n + 1), 2.8))
    axes[0].imshow(img)
    axes[0].set_title("original", fontsize=9)
    for i, ax in enumerate(axes[1:]):
        ax.imshow(variants[i])
        ax.set_title(f"aug {i+1}", fontsize=9)
    for ax in axes:
        ax.axis("off")
    fig.suptitle("On-the-fly augmentation: flips, rotation, zoom, shift, brightness, contrast")
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)


def plot_class_balance(y_train: np.ndarray, weights: Dict[int, float], path: Path,
                       y_after: Optional[np.ndarray] = None) -> None:
    """Training-set class counts with the class weight annotated above each bar."""
    counts = [int((y_train == c).sum()) for c in range(5)]
    fig, ax = plt.subplots(figsize=(7.5, 4))
    x = np.arange(5)
    if y_after is not None:
        after = [int((y_after == c).sum()) for c in range(5)]
        ax.bar(x - 0.2, counts, width=0.4, color="#264653", label="before over-sampling")
        ax.bar(x + 0.2, after, width=0.4, color="#e9c46a", label="after over-sampling")
        ax.legend()
    else:
        ax.bar(x, counts, color=PALETTE)
    for i, c in enumerate(counts):
        ax.text(i, max(counts) * 1.03, f"w = {weights.get(i, 1):.2f}", ha="center", fontsize=9)
    ax.set_xticks(x)
    ax.set_xticklabels(C.CLASS_LABELS)
    ax.set_ylabel("Training images")
    ax.set_ylim(0, max(counts) * 1.18)
    ax.set_title("Training-set imbalance and the class weights used in the loss")
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)


def plot_experiment_comparison(results: pd.DataFrame, path: Path,
                               metrics=("accuracy", "macro_f1", "quadratic_weighted_kappa", "referable_sensitivity")) -> None:
    """Bar chart comparing several runs (backbones / preprocessing methods)."""
    d = results.melt(id_vars="run_name", value_vars=list(metrics), var_name="metric", value_name="score")
    fig, ax = plt.subplots(figsize=(max(8, 1.6 * len(results)), 4.2))
    sns.barplot(data=d, x="run_name", y="score", hue="metric", ax=ax,
                palette=["#264653", "#2a9d8f", "#e9c46a", "#e76f51"])
    ax.set_ylim(0, 1.05)
    ax.set_xlabel("")
    ax.set_title("Experiment comparison (test set)")
    plt.setp(ax.get_xticklabels(), rotation=20, ha="right")
    for container in ax.containers:
        ax.bar_label(container, fmt="%.2f", fontsize=7, padding=1)
    ax.legend(loc="lower right", fontsize=8)
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)

### `dr_detection/train.py`
Two-phase training with callbacks and the end-to-end run_experiment() function.

In [ ]:
%%writefile src/dr_detection/train.py
"""
Two-phase transfer-learning training strategy.

Phase 1 - *feature extraction*
    The ImageNet backbone is frozen; only the new classification head is
    trained (Adam, lr = 1e-4).  This lets the randomly initialised head settle
    without destroying the pretrained filters.

Phase 2 - *fine-tuning*
    The top ``unfreeze_layers`` layers of the backbone are un-frozen and the
    whole network is trained with a 10x smaller learning rate (1e-5), so the
    generic ImageNet features are gently adapted to retinal texture.

Over-fitting control / experimental design
    * validation split monitored every epoch
    * ``EarlyStopping`` on validation loss (restores the best weights)
    * ``ReduceLROnPlateau`` halves the learning rate when validation loss stalls
    * ``ModelCheckpoint`` keeps the best model on disk
    * ``CSVLogger`` writes a per-epoch log used to plot the curves
    * dropout in the head, on-the-fly augmentation and class weights
    * the test split is evaluated exactly once, after training
"""

from __future__ import annotations

import time
from pathlib import Path
from typing import Dict, Optional

import numpy as np
import pandas as pd

from . import config as C
from .augmentation import build_augmentation, compute_class_weights, oversample_minority, balance_summary
from .data import make_tf_dataset, prepare_data
from .model import build_model, compile_model, count_params, set_fine_tuning
from .utils import ensure_dir, hardware_info, save_json, set_seed, timer


# --------------------------------------------------------------------------- #
# Callbacks
# --------------------------------------------------------------------------- #
def get_callbacks(cfg: C.Config, run_dir: Path, phase: int):
    """Standard callback set for one training phase."""
    from tensorflow import keras

    ckpt = run_dir / f"checkpoint_phase{phase}.keras"
    return [
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=cfg.early_stopping_patience,
            restore_best_weights=True,
            verbose=1,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=cfg.reduce_lr_patience,
            min_lr=1e-7,
            verbose=1,
        ),
        keras.callbacks.ModelCheckpoint(
            filepath=str(ckpt),
            monitor="val_loss",
            save_best_only=True,
            verbose=0,
        ),
        keras.callbacks.CSVLogger(str(run_dir / f"log_phase{phase}.csv"), append=False),
    ]


def _history_to_frame(history, phase: int, epoch_offset: int) -> pd.DataFrame:
    df = pd.DataFrame(history.history)
    df.insert(0, "epoch", np.arange(1, len(df) + 1) + epoch_offset)
    df.insert(1, "phase", phase)
    return df


# --------------------------------------------------------------------------- #
# Training
# --------------------------------------------------------------------------- #
def train_two_phase(model, base, train_ds, val_ds, cfg: C.Config, run_dir: Path,
                    class_weights: Optional[Dict[int, float]] = None) -> pd.DataFrame:
    """Run phase 1 then phase 2 and return the concatenated epoch history."""
    epochs1 = 1 if cfg.quick_test else cfg.epochs_phase1
    epochs2 = 1 if cfg.quick_test else cfg.epochs_phase2

    # ---- phase 1 ------------------------------------------------------------
    print(f"\n===== Phase 1: frozen backbone, lr={cfg.lr_phase1}, epochs={epochs1} =====")
    compile_model(model, cfg.lr_phase1)
    print("params:", count_params(model))
    h1 = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epochs1,
        class_weight=class_weights,
        callbacks=get_callbacks(cfg, run_dir, phase=1),
        shuffle=False,  # the tf.data pipeline already shuffles
        verbose=1,
    )
    hist = _history_to_frame(h1, 1, 0)

    # ---- phase 2 ------------------------------------------------------------
    n_train, n_frozen = set_fine_tuning(base, cfg.unfreeze_layers)
    print(f"\n===== Phase 2: fine-tune top {n_train} layers ({n_frozen} frozen), "
          f"lr={cfg.lr_phase2}, epochs={epochs2} =====")
    compile_model(model, cfg.lr_phase2)  # re-compile so trainable flags take effect
    print("params:", count_params(model))
    h2 = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epochs2,
        class_weight=class_weights,
        callbacks=get_callbacks(cfg, run_dir, phase=2),
        shuffle=False,
        verbose=1,
    )
    hist = pd.concat([hist, _history_to_frame(h2, 2, len(hist))], ignore_index=True)
    hist.to_csv(run_dir / "history.csv", index=False)
    return hist


def run_experiment(cfg: C.Config, data: Optional[dict] = None, save_as_best: bool = False,
                   make_figures: bool = True) -> dict:
    """End-to-end experiment: data -> model -> two-phase training -> evaluation.

    Parameters
    ----------
    cfg          : the run configuration
    data         : optional pre-loaded output of ``data.prepare_data`` (re-used
                   across runs with the same preprocessing to save time)
    save_as_best : also copy the trained model + metadata to ``cfg.models_dir``
                   as ``best_model.keras`` / ``model_metadata.json`` (used by the app)
    make_figures : produce evaluation and Grad-CAM figures

    Returns the metrics dictionary that was written to ``run_dir/metrics.json``.
    """
    from .evaluate import evaluate_model, plot_training_curves
    from .gradcam import gradcam_grid

    set_seed(cfg.seed)
    run_dir = ensure_dir(cfg.run_dir)
    save_json(cfg.to_dict(), run_dir / "config.json")
    print(f"[run] {cfg.run_name}  ->  {run_dir}")

    # ---- data -----------------------------------------------------------
    if data is None:
        data = prepare_data(cfg)
    X_train, y_train = data["X_train"], data["y_train"]
    X_val, y_val = data["X_val"], data["y_val"]
    X_test, y_test = data["X_test"], data["y_test"]

    y_train_before = y_train.copy()
    if cfg.oversample_minority:
        X_train, y_train = oversample_minority(X_train, y_train, cfg.seed)
    class_weights = compute_class_weights(y_train) if cfg.use_class_weights else None
    save_json(balance_summary(y_train_before, y_train if cfg.oversample_minority else None,
                              class_weights or {}), run_dir / "class_balance.json")
    print("[run] class weights:", class_weights)

    aug = build_augmentation(cfg.seed) if cfg.use_augmentation else None
    train_ds = make_tf_dataset(X_train, y_train, cfg.batch_size, shuffle=True, augmentation=aug, seed=cfg.seed)
    val_ds = make_tf_dataset(X_val, y_val, cfg.batch_size)

    # ---- model ----------------------------------------------------------
    model, base = build_model(cfg)
    with open(run_dir / "model_summary.txt", "w", encoding="utf-8") as f:
        model.summary(print_fn=lambda s, **kw: f.write(s + "\n"))

    # ---- train ----------------------------------------------------------
    t0 = time.perf_counter()
    with timer("training"):
        hist = train_two_phase(model, base, train_ds, val_ds, cfg, run_dir, class_weights)
    train_minutes = (time.perf_counter() - t0) / 60
    if make_figures:
        plot_training_curves(hist, run_dir / "training_curves.png", title=cfg.run_name)

    # ---- evaluate -------------------------------------------------------
    metrics = evaluate_model(model, X_test, y_test, run_dir, batch_size=cfg.batch_size,
                             make_figures=make_figures)
    metrics.update({
        "run_name": cfg.run_name,
        "backbone": cfg.backbone,
        "preprocess": cfg.preprocess,
        "img_size": cfg.img_size,
        "epochs_trained": int(len(hist)),
        "best_val_loss": float(hist["val_loss"].min()),
        "best_val_accuracy": float(hist["val_accuracy"].max()),
        "train_minutes": round(train_minutes, 2),
        "params": count_params(model),
        "hardware": hardware_info(),
    })
    save_json(metrics, run_dir / "metrics.json")

    # ---- save model + Grad-CAM -------------------------------------------
    model_path = run_dir / "model.keras"
    model.save(model_path)
    if make_figures:
        try:
            gradcam_grid(model, X_test, y_test, run_dir / "gradcam_test_samples.png", n_per_class=2)
        except Exception as exc:  # Grad-CAM must never break a run
            print("[run] Grad-CAM failed:", exc)

    if save_as_best:
        ensure_dir(cfg.models_dir)
        model.save(Path(cfg.models_dir) / "best_model.keras")
        save_json({
            "run_name": cfg.run_name,
            "backbone": cfg.backbone,
            "preprocess": cfg.preprocess,
            "img_size": cfg.img_size,
            "class_names": C.CLASS_NAMES,
            "class_labels": C.CLASS_LABELS,
            "metrics": {k: metrics[k] for k in ("accuracy", "macro_f1", "weighted_f1",
                                                 "quadratic_weighted_kappa",
                                                 "referable_sensitivity", "referable_specificity")
                        if k in metrics},
            "trained_on": metrics["hardware"],
        }, Path(cfg.models_dir) / "model_metadata.json")
        print(f"[run] saved best model to {cfg.models_dir}")

    print(f"[run] done: acc={metrics['accuracy']:.4f}  macro-F1={metrics['macro_f1']:.4f}  "
          f"QWK={metrics['quadratic_weighted_kappa']:.4f}")
    return metrics

In [ ]:
# make the freshly written package importable (and re-importable after edits)
sys.path.insert(0, str(WORK_DIR / "src"))
importlib.invalidate_caches()
for m in [m for m in sys.modules if m.startswith("dr_detection")]:
    del sys.modules[m]
import dr_detection
print("dr_detection", dr_detection.__version__, "imported from", Path(dr_detection.__file__).parent)

## 3. Dataset: APTOS 2019 Blindness Detection (224 x 224 version)

**Source:** Kaggle dataset `sovitrath/diabetic-retinopathy-224x224-2019-data` (CC0 licence).
It is the training set of the *APTOS 2019 Blindness Detection* competition (Aravind Eye Hospital, India),
resized to 224 x 224 px by the dataset author: **3,662 colour fundus photographs** graded by clinicians on
the International Clinical Diabetic Retinopathy (ICDR) scale.

| Grade | Folder | Meaning |
|---|---|---|
| 0 | `No_DR` | no diabetic retinopathy |
| 1 | `Mild` | microaneurysms only |
| 2 | `Moderate` | more than microaneurysms, less than severe |
| 3 | `Severe` | extensive haemorrhages, venous beading or IRMA |
| 4 | `Proliferate_DR` | neovascularisation or vitreous haemorrhage |

Note the folder spelling `Proliferate_DR`. A naive label dictionary keyed on `Proliferative_DR` silently
drops those 295 images; our `normalise_label()` accepts every spelling, so no image is lost.

In [ ]:
# ============================================================================
# 3.1  Locate the attached dataset and show its structure (dataset evidence)
# ============================================================================
from dr_detection import config as C
from dr_detection.data import find_dataset_dir

DATA_DIR = find_dataset_dir(INPUT_DIR)
print("Kaggle dataset :", C.KAGGLE_DATASET)
print("Mounted at     :", DATA_DIR)
for p in sorted(DATA_DIR.iterdir()):
    if p.is_dir():
        for q in sorted(p.iterdir()):
            if q.is_dir():
                print(f"   {p.name}/{q.name:<16s} {len(list(q.glob('*.png'))):>5d} images")
    else:
        print(f"   {p.name:<30s} {p.stat().st_size/1024:.0f} KB")

In [ ]:
# ============================================================================
# 3.2  Configuration of the MAIN experiment (all hyper-parameters in one place)
# ============================================================================
from dr_detection.utils import set_seed, save_json

cfg = C.Config(
    data_dir=str(DATA_DIR),
    results_dir=str(WORK_DIR / "results"),
    models_dir=str(WORK_DIR / "models"),
    cache_dir=str(CACHE_DIR),
    backbone="EfficientNetB0",      # main model; section 8 compares 4 alternatives
    preprocess="ben_graham",        # main preprocessing; section 8 compares 3 alternatives
    img_size=224, batch_size=32,
    epochs_phase1=10, epochs_phase2=20,
    lr_phase1=1e-4, lr_phase2=1e-5, unfreeze_layers=30,
    dropout=0.3, use_class_weights=True, use_augmentation=True,
    early_stopping_patience=6, reduce_lr_patience=3, seed=42,
    quick_test=QUICK_TEST, quick_test_per_class=20,
)
set_seed(cfg.seed)
FIG_DIR = Path(cfg.results_dir) / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
pd.DataFrame(cfg.to_dict().items(), columns=["parameter", "value"])

In [ ]:
# ============================================================================
# 3.3  Build the DataFrame and cross-check the labels against train.csv
# ============================================================================
from dr_detection.data import build_dataframe, cross_check_with_csv, class_distribution, split_dataframe

df = build_dataframe(cfg.images_dir)
df = cross_check_with_csv(df, cfg.data_dir)
print("Total images:", len(df))
class_distribution(df)

In [ ]:
# ============================================================================
# 3.4  Stratified 70 / 10 / 20 split (the same seed is used by every experiment)
# ============================================================================
df_train, df_val, df_test = split_dataframe(df, cfg.val_frac, cfg.test_frac, cfg.seed)
split_table = pd.DataFrame({
    "train": df_train["label"].value_counts().sort_index(),
    "validation": df_val["label"].value_counts().sort_index(),
    "test": df_test["label"].value_counts().sort_index(),
}).rename(index=lambda i: f"{i} - {C.CLASS_LABELS[i]}")
split_table["total"] = split_table.sum(axis=1)
split_table.loc["TOTAL"] = split_table.sum()
split_table

In [ ]:
# ============================================================================
# 3.5  EDA figures: class distribution, split, samples, illumination variability
# ============================================================================
from dr_detection import visualize as V
from IPython.display import Image as IImage, HTML, display

def show(path, width=850, title=None):
    """Display a saved figure inline (the same PNG goes into the report)."""
    if title:
        display(HTML(f"<h4>{title}</h4>"))
    display(IImage(filename=str(path), width=width))

V.plot_class_distribution(df, FIG_DIR / "class_distribution.png")
V.plot_split_distribution(df_train, df_val, df_test, FIG_DIR / "split_distribution.png")
V.plot_sample_grid(df, FIG_DIR / "sample_grid_raw.png", method="raw", seed=cfg.seed)
intensity = V.plot_intensity_stats(df, FIG_DIR / "intensity_stats.png", seed=cfg.seed)
for f in ["class_distribution.png", "split_distribution.png", "sample_grid_raw.png", "intensity_stats.png"]:
    show(FIG_DIR / f, title=f)

## 4. Preprocessing

Every image goes through **black-border crop, resize to 224 px, then contrast and edge enhancement**.

* **Ben Graham** (`4*img - 4*GaussianBlur(img, sigma=10) + 128`) removes illumination gradients and colour
  casts between cameras and amplifies local contrast, so microaneurysms, haemorrhages and exudates stand out.
  A circular mask removes the artificial grey frame the subtraction creates.
* **CLAHE** on the L channel of the LAB colour space boosts local contrast without shifting colours.
* **Unsharp masking** and the **Canny edge map** illustrate what edge enhancement emphasises (vessels, lesion borders).

Section 8 measures which method actually helps the classifier.

In [ ]:
example = df[df["label"] == 2].sample(1, random_state=cfg.seed)["image_path"].iloc[0]
V.plot_preprocessing_stages(example, FIG_DIR / "preprocessing_stages.png")
V.plot_preprocessing_comparison(df, FIG_DIR / "preprocessing_comparison.png", seed=cfg.seed)
V.plot_sample_grid(df, FIG_DIR / "sample_grid_ben_graham.png", method="ben_graham", seed=cfg.seed)
show(FIG_DIR / "preprocessing_stages.png", 1100, "One image through every preprocessing stage")
show(FIG_DIR / "preprocessing_comparison.png", 700, "Each preprocessing method on one image per grade")
show(FIG_DIR / "sample_grid_ben_graham.png", 700, "Samples after Ben Graham preprocessing")

In [ ]:
# Preprocess the whole dataset once into RAM (cached as .npz in /tmp so re-runs are instant)
from dr_detection.data import prepare_data
t0 = time.time()
data = prepare_data(cfg)
print(f"preprocessing took {time.time() - t0:.0f} s")
{k: v.shape for k, v in data.items() if k.startswith(("X_", "y_"))}

## 5. Data augmentation and class balancing

Augmentation runs **on the GPU, on the fly, on the training split only**: random horizontal and vertical flips,
rotation (up to 36 degrees), zoom (10 %), translation (5 %), brightness and contrast (10 %). Every epoch sees a
different variant of each image, which fights over-fitting better than a fixed set of pre-generated copies.
Validation and test images are never augmented.

The dataset is imbalanced (about 49 % No DR vs 5 % Severe). **Inverse-frequency class weights**
(`weight_c = N / (5 * n_c)`) make a mistake on a rare severe case cost several times more than one on a healthy
eye. Random over-sampling is also implemented (`Config(oversample_minority=True)`).

In [ ]:
from dr_detection.augmentation import compute_class_weights
weights = compute_class_weights(data["y_train"])
V.plot_augmentation_grid(example, FIG_DIR / "augmentation_grid.png", method=cfg.preprocess)
V.plot_class_balance(data["y_train"], weights, FIG_DIR / "class_balance.png")
show(FIG_DIR / "augmentation_grid.png", 1100, "Random augmentations of one preprocessed image")
show(FIG_DIR / "class_balance.png", 700, "Training-set imbalance and class weights")
pd.DataFrame({"class": C.CLASS_LABELS,
              "train images": [int((data["y_train"] == c).sum()) for c in range(5)],
              "class weight": [round(weights[c], 3) for c in range(5)]})

## 6. CNN architecture, transfer learning and two-phase training

```
Input 224x224x3 (0-255) -> backbone normalisation -> EfficientNetB0 (ImageNet weights, include_top=False)
  -> GlobalAveragePooling -> Dropout 0.3 -> Dense 512 ReLU -> Dropout 0.3 -> Dense 256 ReLU -> Dense 5 softmax
```

* **Phase 1, feature extraction:** backbone frozen, only the new head trains (Adam, lr 1e-4, up to 10 epochs).
* **Phase 2, fine-tuning:** the top 30 backbone layers are unfrozen with BatchNorm kept frozen (Adam, lr 1e-5, up to 20 epochs).
* **Callbacks:** EarlyStopping on validation loss (patience 6, best weights restored), ReduceLROnPlateau (halve lr, patience 3),
  ModelCheckpoint, CSVLogger.
* **Loss:** sparse categorical cross-entropy weighted by the class weights above.

In [ ]:
from dr_detection.model import build_model, count_params
_model, _base = build_model(cfg)
print("backbone layers:", len(_base.layers), "| parameters:", count_params(_model))
_model.summary(show_trainable=True)
del _model, _base
tf.keras.backend.clear_session()

In [ ]:
# ============================================================================
# Train and evaluate the MAIN model (about 5-10 min on a T4).
# Writes results/<run_name>/ and models/best_model.keras for the Streamlit app.
# ============================================================================
from dr_detection.train import run_experiment
metrics = run_experiment(cfg, data=data, save_as_best=True)
RUN = cfg.run_dir

## 7. Evaluation of the main model on the held-out test set

In [ ]:
hist = pd.read_csv(RUN / "history.csv")
print(f"Epochs trained: {len(hist)} (phase 1: {(hist.phase == 1).sum()}, phase 2: {(hist.phase == 2).sum()})")
show(RUN / "training_curves.png", 1100, "Accuracy and loss curves")
hist.round(4)

In [ ]:
print(metrics["classification_report"])
headline = ["accuracy", "macro_precision", "macro_recall", "macro_f1", "weighted_f1", "quadratic_weighted_kappa",
            "cohen_kappa", "macro_auc_ovr", "within_one_grade_accuracy", "mean_absolute_grade_error",
            "referable_sensitivity", "referable_specificity"]
pd.DataFrame([(k, metrics[k]) for k in headline if k in metrics], columns=["metric", "test value"]).round(4)

In [ ]:
for f, title in [("confusion_matrix.png", "Confusion matrix"),
                 ("confusion_matrix_normalised.png", "Normalised confusion matrix (rows = true grade)"),
                 ("per_class_metrics.png", "Per-class precision, recall and F1"),
                 ("roc_curves.png", "ROC curves"),
                 ("prediction_distribution.png", "True vs predicted grade distribution"),
                 ("misclassified.png", "Error analysis: most confident mistakes"),
                 ("gradcam_test_samples.png", "Grad-CAM explanations on test images")]:
    if (RUN / f).exists():
        show(RUN / f, 850, title)

In [ ]:
# Clinically meaningful binary views: referable DR (grade >= 2) and any DR (grade >= 1)
pd.DataFrame({"referable DR (grade >= 2)": metrics["referable_dr"],
              "any DR (grade >= 1)": metrics["any_dr"]}).round(4)

## 8. Experiments: preprocessing methods and CNN backbones

Same split, same seed, same training recipe; only one factor changes per run.

* **Preprocessing study:** EfficientNetB0 with raw, CLAHE, Ben Graham, and CLAHE + Ben Graham images.
* **Backbone study:** Ben Graham images with EfficientNetB0, EfficientNetB3, ResNet50V2, MobileNetV2 and DenseNet121.

Set `RUN_COMPARISON = False` to skip (saves about 60 to 90 GPU minutes). Each run is isolated in `try/except`,
so one failure (for example running out of GPU memory) never stops the notebook, and finished runs are skipped on a re-run.

In [ ]:
RUN_COMPARISON = True          # <-- set False to skip the grid
import dataclasses, gc, traceback

PREPROCESS_GRID = ["raw", "clahe", "ben_graham", "clahe_ben_graham"]
BACKBONE_GRID   = ["EfficientNetB0", "EfficientNetB3", "ResNet50V2", "MobileNetV2", "DenseNet121"]
if QUICK_TEST:                 # local smoke test only
    PREPROCESS_GRID, BACKBONE_GRID = ["raw", "ben_graham"], ["EfficientNetB0", "MobileNetV2"]

if RUN_COMPARISON:
    grid = [("EfficientNetB0", p) for p in PREPROCESS_GRID] + [(b, "ben_graham") for b in BACKBONE_GRID]
    grid = list(dict.fromkeys(grid))
    data_cache = {cfg.preprocess: data}
    for backbone, prep in grid:
        run_cfg = dataclasses.replace(cfg, backbone=backbone, preprocess=prep, run_name="")
        if (run_cfg.run_dir / "metrics.json").exists():
            print("skip (already done):", run_cfg.run_name)
            continue
        try:
            if prep not in data_cache:
                # keep only the main preprocessing plus the current one in RAM
                for k in [k for k in data_cache if k != cfg.preprocess]:
                    del data_cache[k]
                gc.collect()
                data_cache[prep] = prepare_data(run_cfg)
            t0 = time.time()
            run_experiment(run_cfg, data=data_cache[prep], make_figures=True)
            print(f"finished {run_cfg.run_name} in {(time.time() - t0) / 60:.1f} min")
        except Exception:
            print(f"!!! run {run_cfg.run_name} failed, continuing with the next one")
            traceback.print_exc()
        tf.keras.backend.clear_session()
        gc.collect()

In [ ]:
from dr_detection.visualize import plot_experiment_comparison
cols = ["run_name", "backbone", "preprocess", "accuracy", "macro_precision", "macro_recall", "macro_f1", "weighted_f1",
        "quadratic_weighted_kappa", "referable_sensitivity", "referable_specificity", "within_one_grade_accuracy",
        "epochs_trained", "train_minutes"]
rows = []
for mfile in sorted(Path(cfg.results_dir).glob("*/metrics.json")):
    m = json.loads(mfile.read_text())
    rows.append({k: m.get(k) for k in cols} | {"total_params": m.get("params", {}).get("total")})
table = pd.DataFrame(rows).sort_values("quadratic_weighted_kappa", ascending=False).reset_index(drop=True)
table.to_csv(Path(cfg.results_dir) / "experiments_comparison.csv", index=False)
if len(table) > 1:
    plot_experiment_comparison(table, Path(cfg.results_dir) / "experiments_comparison.png")
    show(Path(cfg.results_dir) / "experiments_comparison.png", 1100, "Experiment comparison")
table.round(4)

## 9. Export the results bundle

The bundle contains everything the report and the Streamlit demo need:

* `results/`: metrics, histories, classification reports and every figure (per-run model files excluded to keep it small)
* `models/best_model.keras` and `models/model_metadata.json`: the main model used by the demo app
* `app/assets/samples/`: two real **test-set** images per grade, with the true grade in the file name, for the video

Download it from this version's **Output** tab, then unzip it into the root of the GitHub repository.

In [ ]:
import zipfile

samples_dir = WORK_DIR / "app" / "assets" / "samples"
samples_dir.mkdir(parents=True, exist_ok=True)
for c in range(5):
    pool = data["df_test"][data["df_test"]["label"] == c]
    for _, row in pool.sample(min(2, len(pool)), random_state=1).iterrows():
        shutil.copy(row["image_path"], samples_dir / f"grade{c}_{C.CLASS_NAMES[c]}_{row['id_code']}.png")

# headline numbers for the report, in one small file
report_numbers = {
    "kaggle_dataset": C.KAGGLE_DATASET,
    "total_images": int(len(df)),
    "split_sizes": {"train": int(len(df_train)), "val": int(len(df_val)), "test": int(len(df_test))},
    "class_weights": weights,
    "main_run": cfg.run_name,
    "main_metrics": {k: metrics[k] for k in headline if k in metrics},
    "referable_dr": metrics["referable_dr"],
    "comparison": table.round(4).to_dict(orient="records"),
    "environment": {"tensorflow": tf.__version__, "keras": keras.__version__,
                    "gpu": [tf.config.experimental.get_device_details(g).get("device_name", g.name) for g in gpus]},
    "notebook_minutes": round((time.time() - NOTEBOOK_START) / 60, 1),
}
save_json(report_numbers, Path(cfg.results_dir) / "report_numbers.json")

bundle = WORK_DIR / "dr_results_bundle.zip"
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in Path(cfg.results_dir).rglob("*"):
        if p.is_file() and p.suffix != ".keras":
            zf.write(p, p.relative_to(WORK_DIR))
    for p in list(Path(cfg.models_dir).glob("*")) + list(samples_dir.glob("*.png")):
        zf.write(p, p.relative_to(WORK_DIR))

# per-run model files are large and already summarised in the bundle; remove them from the output
for p in Path(cfg.results_dir).glob("*/*.keras"):
    p.unlink()
print(f"bundle: {bundle} ({bundle.stat().st_size / 1e6:.1f} MB)")
print(f"total notebook time: {(time.time() - NOTEBOOK_START) / 60:.1f} min")

---
### Next steps outside Kaggle
1. Download `dr_results_bundle.zip` from the **Output** tab and unzip it into the repository root.
2. `pip install -r requirements.txt`, then `streamlit run app/streamlit_app.py` to launch the demo for the video.
3. Every number in the report comes from `results/report_numbers.json` and `results/<run>/metrics.json`.